# STUDY ASSISTANT v2.0 - Wk10
# PariShiksha | NCERT Gravitation (iesc110)

In [1]:
import os

folders = ['data', 'outputs']
files = [
    'wk10_chunks.json',
    'chunking_diff.md',
    'retrieval_log.json',
    'retrieval_misses.md',
    'prompt_diff.md',
    'eval_scored.csv',
    'eval_v2_scored.csv',
    'fix_memo.md',
    'reflection.md',
    'README.md'
]

In [2]:
for folder in folders:
    os.makedirs(folder, exist_ok=True)
    print(f"Created folder: {folder}")

for file in files:
    if not os.path.exists(file):
        open(file, 'w').close()
        print(f"Created file: {file}")
    else:
        print(f"Already exists: {file}")

print("\n Project structure ready!")

Created folder: data
Created folder: outputs
Already exists: wk10_chunks.json
Already exists: chunking_diff.md
Already exists: retrieval_log.json
Already exists: retrieval_misses.md
Already exists: prompt_diff.md
Already exists: eval_scored.csv
Already exists: eval_v2_scored.csv
Already exists: fix_memo.md
Already exists: reflection.md
Already exists: README.md

 Project structure ready!


In [3]:
import sys
!{sys.executable} -m pip install PyMuPDF

In [4]:
import fitz

doc = fitz.open("data/iesc110.pdf")

print(f"Total pages: {len(doc)}")
print(f"\n--- Checking all page headings ---\n")

for i, page in enumerate(doc):
    text = page.get_text()
    preview = text[:150].replace('\n', ' ')
    print(f"Page {i+1}: {preview}")
    print()

Total pages: 24

--- Checking all page headings ---

Page 1: Sound Waves:  Characteristics and  Applications Chapter  10 Sound is an everyday sensory experience that helps us become  aware of our surroundings. E

Page 2: Sound Waves: Characteristics and Applications 185 Activity 10.1: Let us explore 1.	 Take a cardboard box with one side open  and a rubber band. 2.	 St

Page 3: 186 Exploration|Grade 9 10.1.1 Tuning fork An instrument that is often used for experiments with sound is a tuning  fork. A tuning fork is a U-shaped 

Page 4: Sound Waves: Characteristics and Applications 187 2.	 Now, place your ear against the desk, close  your other ear and listen again, as shown  in Fig. 

Page 5: 188 Exploration|Grade 9 3.	 Assertion (A): We cannot hear the sound of a bell ringing in a closed jar  after most of the air is pumped out. 	 Reason (

Page 6: Sound Waves: Characteristics and Applications 189 Fig. 10.9: Density of air in a tube with a piston (a) Piston not oscillating (average air

# STAGE 1 — Chunking
# Goal: Split iesc110 PDF into metadata-rich chunks

In [5]:
import fitz          # PDF extraction
import json          # saving chunks
import re            # pattern matching for content types
import tiktoken      # token counting
from collections import Counter

print("All libraries imported successfully")

All libraries imported successfully


In [6]:
tokenizer = tiktoken.get_encoding("cl100k_base")

def count_tokens(text):
    return len(tokenizer.encode(text))

# Quick test
test = "Sound is a form of energy that travels through a medium."
print(f"Test sentence: '{test}'")
print(f"Token count: {count_tokens(test)}")
print("Tokenizer ready")

Test sentence: 'Sound is a form of energy that travels through a medium.'
Token count: 12
Tokenizer ready


In [7]:
def get_content_type(text):
    text_lower = text.lower().strip()
    
    
    if re.search(r'(^example\s+\d|example\s+\d+\.|worked example|solution\s*:|let us calculate)', text_lower):
        return "worked_example"
    
    
    elif re.search(r'(activity\s+\d+|exercise\s+\d+|\d+\.\s+which|assertion\s*\(a\)|try this|do you know)', text_lower):
        return "question_or_exercise"
    
    else:
        return "prose"

# Test on 3 sentences
tests = [
    ("Example 10.1: Calculate the wavelength.", "worked_example"),
    ("Activity 10.1: Take a cardboard box.", "question_or_exercise"),
    ("Sound travels faster in solids than gases.", "prose"),
    ("For example, grasshoppers rub their wings.", "prose"),  
    ("Sound Waves: Characteristics and Applications Chapter 10", "prose"), 
]

print("--- Classifier test results ---\n")
all_passed = True
for text, expected in tests:
    result = get_content_type(text)
    status = "Yes" if result == expected else "No"
    if result != expected:
        all_passed = False
    print(f"{status} Expected: {expected}")
    print(f"   Got:      {result}")
    print(f"   Text:     '{text[:60]}'")
    print()

if all_passed:
    print("All tests passed! Classifier improved.")
else:
    print("Some tests still failing — let's fix further.")

--- Classifier test results ---

Yes Expected: worked_example
   Got:      worked_example
   Text:     'Example 10.1: Calculate the wavelength.'

Yes Expected: question_or_exercise
   Got:      question_or_exercise
   Text:     'Activity 10.1: Take a cardboard box.'

Yes Expected: prose
   Got:      prose
   Text:     'Sound travels faster in solids than gases.'

Yes Expected: prose
   Got:      prose
   Text:     'For example, grasshoppers rub their wings.'

Yes Expected: prose
   Got:      prose
   Text:     'Sound Waves: Characteristics and Applications Chapter 10'

All tests passed! Classifier improved.


In [8]:
def chunk_pdf(pdf_path, max_tokens=250):
    doc = fitz.open(pdf_path)
    chunks = []
    chunk_id = 0

    for page_num, page in enumerate(doc):
        text = page.get_text()

        # Split page into paragraphs
        paragraphs = [p.strip() for p in text.split('\n\n') if p.strip()]

        current_chunk = ""
        current_tokens = 0

        for para in paragraphs:
            para_tokens = count_tokens(para)

            # If paragraph itself is too big, split by sentences
            if para_tokens > max_tokens:
                sentences = re.split(r'(?<=[.!?])\s+', para)
                for sentence in sentences:
                    sent_tokens = count_tokens(sentence)
                    if current_tokens + sent_tokens > max_tokens:
                        if current_chunk.strip():
                            chunks.append({
                                "chunk_id": f"chunk_{chunk_id:04d}",
                                "source": pdf_path,
                                "page": page_num + 1,
                                "content_type": get_content_type(current_chunk),
                                "text": current_chunk.strip(),
                                "token_count": current_tokens
                            })
                            chunk_id += 1
                        current_chunk = sentence
                        current_tokens = sent_tokens
                    else:
                        current_chunk += " " + sentence
                        current_tokens += sent_tokens
            else:
                # Paragraph fits — add to current chunk or start new one
                if current_tokens + para_tokens > max_tokens:
                    if current_chunk.strip():
                        chunks.append({
                            "chunk_id": f"chunk_{chunk_id:04d}",
                            "source": pdf_path,
                            "page": page_num + 1,
                            "content_type": get_content_type(current_chunk),
                            "text": current_chunk.strip(),
                            "token_count": current_tokens
                        })
                        chunk_id += 1
                    current_chunk = para
                    current_tokens = para_tokens
                else:
                    current_chunk += "\n\n" + para
                    current_tokens += para_tokens

        # Save last chunk of the page
        if current_chunk.strip():
            chunks.append({
                "chunk_id": f"chunk_{chunk_id:04d}",
                "source": pdf_path,
                "page": page_num + 1,
                "content_type": get_content_type(current_chunk),
                "text": current_chunk.strip(),
                "token_count": current_tokens
            })
            chunk_id += 1

    return chunks

print("Chunking function defined and ready")

Chunking function defined and ready


In [9]:
chunks = chunk_pdf("data/iesc110.pdf")

print(f"Total chunks created: {len(chunks)}")
print(f"\n Content type breakdown")
types = Counter(c['content_type'] for c in chunks)
for t, count in types.items():
    print(f"  {t}: {count}")

print(f"\n Token size stats")
token_counts = [c['token_count'] for c in chunks]
print(f"  Min tokens: {min(token_counts)}")
print(f"  Max tokens: {max(token_counts)}")
print(f"  Avg tokens: {sum(token_counts)//len(token_counts)}")

Total chunks created: 69

 Content type breakdown
  prose: 50
  question_or_exercise: 13
  worked_example: 6

 Token size stats
  Min tokens: 24
  Max tokens: 250
  Avg tokens: 191


In [10]:
for content_type in ['prose', 'worked_example', 'question_or_exercise']:
    print(f"\n{'='*50}")
    print(f"CONTENT TYPE: {content_type}")
    print(f"{'='*50}")
    
    for c in chunks:
        if c['content_type'] == content_type:
            print(f"chunk_id : {c['chunk_id']}")
            print(f"page     : {c['page']}")
            print(f"tokens   : {c['token_count']}")
            print(f"text preview:\n{c['text'][:250]}")
            break


CONTENT TYPE: prose
chunk_id : chunk_0000
page     : 1
tokens   : 191
text preview:
Sound Waves: 
Characteristics and 
Applications
Chapter 
10
Sound is an everyday sensory experience that helps us become 
aware of our surroundings. Every day, we hear a variety of sounds 
in our surroundings, such as human voices, birds chirping, 
w

CONTENT TYPE: worked_example
chunk_id : chunk_0032
page     : 12
tokens   : 231
text preview:
Sound Waves: Characteristics and Applications
195
Density
Distance
Amplitude
Density
Distance
Amplitude
C 
R 
C 
R 
C 
R
C 
R 
C 
R 
C 
R
Fig. 10.20: Sound waves
(a) Low amplitude
(b) High amplitude
In this activity, you would have noticed that the f

CONTENT TYPE: question_or_exercise
chunk_id : chunk_0002
page     : 2
tokens   : 241
text preview:
Sound Waves: Characteristics and Applications
185
Activity 10.1: Let us explore
1. Take a cardboard box with one side open 
and a rubber band. 2. Stretch the rubber band across the open 
side of the box (Fig. 10.2). 3.

In [11]:
with open("wk10_chunks.json", "w", encoding="utf-8") as f:
    json.dump(chunks, f, indent=2, ensure_ascii=False)

# Verify it saved correctly
with open("wk10_chunks.json", "r", encoding="utf-8") as f:
    saved = json.load(f)

print(f"Saved {len(saved)} chunks to wk10_chunks.json")
print(f"\n--- First chunk preview ---")
print(json.dumps(saved[0], indent=2)[:400])

Saved 69 chunks to wk10_chunks.json

--- First chunk preview ---
{
  "chunk_id": "chunk_0000",
  "source": "data/iesc110.pdf",
  "page": 1,
  "content_type": "prose",
  "text": "Sound Waves: \nCharacteristics and \nApplications\nChapter \n10\nSound is an everyday sensory experience that helps us become \naware of our surroundings. Every day, we hear a variety of sounds \nin our surroundings, such as human voices, birds chirping, \nwaves crashing on the seashore


# STAGE 2 — Embeddings and Vector Retrieval
# Goal: Embed chunks into Chroma, retrieve by semantic similarity

In [12]:
import json

with open("wk10_chunks.json", "r", encoding="utf-8") as f:
    chunks = json.load(f)

print(f"Loaded {len(chunks)} chunks from wk10_chunks.json")
print(f"\n--- First chunk preview ---")
print(f"chunk_id : {chunks[0]['chunk_id']}")
print(f"page     : {chunks[0]['page']}")
print(f"type     : {chunks[0]['content_type']}")
print(f"tokens   : {chunks[0]['token_count']}")
print(f"text     : {chunks[0]['text'][:150]}")

Loaded 69 chunks from wk10_chunks.json

--- First chunk preview ---
chunk_id : chunk_0000
page     : 1
type     : prose
tokens   : 191
text     : Sound Waves: 
Characteristics and 
Applications
Chapter 
10
Sound is an everyday sensory experience that helps us become 
aware of our surroundings. E


In [13]:
from sentence_transformers import SentenceTransformer

print("Loading embedding model... (may take 1-2 mins on first run)")
embedder = SentenceTransformer('BAAI/bge-small-en')

# Quick test
test_sentence = "Sound is a form of energy."
embedding = embedder.encode(test_sentence)

print(f"Model loaded successfully!")
print(f"Embedding dimension: {len(embedding)}")
print(f"Test sentence: '{test_sentence}'")
print(f"First 5 values: {embedding[:5]}")

Loading embedding model... (may take 1-2 mins on first run)
Model loaded successfully!
Embedding dimension: 384
Test sentence: 'Sound is a form of energy.'
First 5 values: [-0.006775   -0.0251682   0.03833768 -0.02817889 -0.01124422]


# Set up Chroma vector store

In [15]:
import chromadb

client = chromadb.PersistentClient(path="./chroma_wk10")

collection = client.get_or_create_collection(
    name="sound_waves",
    metadata={"hnsw:space": "cosine"}
)

print(f"Chroma client created at ./chroma_wk10")
print(f"Collection 'sound_waves' ready")
print(f"Documents currently in collection: {collection.count()}")

Failed to send telemetry event ClientStartEvent: capture() takes 1 positional argument but 3 were given
Failed to send telemetry event ClientCreateCollectionEvent: capture() takes 1 positional argument but 3 were given


Chroma client created at ./chroma_wk10
Collection 'sound_waves' ready
Documents currently in collection: 0


In [16]:
if collection.count() == 0:
    print("Collection is empty — embedding all chunks now...")
    print("This may take 1-2 minutes...")
    
    texts = [c['text'] for c in chunks]
    ids = [c['chunk_id'] for c in chunks]
    metadatas = [{
        "page": c['page'],
        "content_type": c['content_type'],
        "token_count": c['token_count']
    } for c in chunks]
    
    # Embed in batches of 10
    batch_size = 10
    for i in range(0, len(chunks), batch_size):
        batch_texts = texts[i:i+batch_size]
        batch_ids = ids[i:i+batch_size]
        batch_meta = metadatas[i:i+batch_size]
        
        embeddings = embedder.encode(batch_texts).tolist()
        
        collection.add(
            documents=batch_texts,
            embeddings=embeddings,
            ids=batch_ids,
            metadatas=batch_meta
        )
        print(f"  Embedded chunks {i+1} to {min(i+batch_size, len(chunks))}...")
    
    print(f"\n All chunks embedded and stored!")

else:
    print(f"Collection already has {collection.count()} chunks — skipping embedding")

print(f"\nTotal documents in Chroma: {collection.count()}")

Collection is empty — embedding all chunks now...
This may take 1-2 minutes...


Failed to send telemetry event CollectionAddEvent: capture() takes 1 positional argument but 3 were given


  Embedded chunks 1 to 10...
  Embedded chunks 11 to 20...
  Embedded chunks 21 to 30...
  Embedded chunks 31 to 40...
  Embedded chunks 41 to 50...
  Embedded chunks 51 to 60...
  Embedded chunks 61 to 69...

 All chunks embedded and stored!

Total documents in Chroma: 69


# Build the retrieve function

In [18]:
def retrieve(query, k=5):
    
    query_embedding = embedder.encode(query).tolist()
    
    
    results = collection.query(
        query_embeddings=[query_embedding],
        n_results=k
    )
    
    # Formatting the results
    retrieved = []
    for i in range(len(results['ids'][0])):
        retrieved.append({
            "rank": i + 1,
            "chunk_id": results['ids'][0][i],
            "score": round(1 - results['distances'][0][i], 4),  # convert distance to similarity
            "content_type": results['metadatas'][0][i]['content_type'],
            "page": results['metadatas'][0][i]['page'],
            "text": results['documents'][0][i]
        })
    
    return retrieved

print("Retrieve function ready")

Retrieve function ready


# Test the retreival

In [19]:
test_queries = [
    "What is the speed of sound in air?",
    "What is an echo?",
    "How do humans hear sound?"
]

for query in test_queries:
    print(f"\n{'='*55}")
    print(f"QUERY: {query}")
    print(f"{'='*55}")
    
    results = retrieve(query, k=3)
    
    for r in results:
        print(f"\nRank {r['rank']} | Score: {r['score']} | Page: {r['page']} | Type: {r['content_type']}")
        print(f"chunk_id: {r['chunk_id']}")
        print(f"Text preview: {r['text'][:200]}")

Failed to send telemetry event CollectionQueryEvent: capture() takes 1 positional argument but 3 were given



QUERY: What is the speed of sound in air?

Rank 1 | Score: 0.9002 | Page: 23 | Type: prose
chunk_id: chunk_0064
Text preview: 206
Exploration|Grade 9
12. The speed of sound in air is about 331 m s–1 at 0 ºC and nearly 
344 m s–1 at 22 ºC. Roughly how much extra time will the sound of 
thunder take to travel a distance of 172

Rank 2 | Score: 0.8902 | Page: 14 | Type: worked_example
chunk_id: chunk_0038
Text preview: Sound Waves: Characteristics and Applications
197
λ
speed of the wave
Therefore, wavelength  =  
frequency
	
(i)	 For ν  = 20 Hz,        λ
−
−
1
1
344 m s
 =  
 = 17.2 m 
20 s
	
(ii)	For ν  = 20 kHz =

Rank 3 | Score: 0.8864 | Page: 13 | Type: prose
chunk_id: chunk_0036
Text preview: Therefore, 
the speed of sound (v ) is
distance
speed =  
time
                           ⇒    v
T
λ
=
We know that frequency 
T
ν = 1  (Eq. 10.1). Thus, 
	
	
	
	
         
λ
ν
×
v =  

(10.2)
speed

QUERY: What is an echo?

Rank 1 | Score: 0.8501 | Page: 18 | Type: prose
chunk_id: chunk_0050


In [20]:
import json

eval_queries = [
    "What is the speed of sound in air?",
    "What is an echo?",
    "How do humans hear sound?",
    "What is the frequency of sound?",
    "What is wavelength of a sound wave?",
    "How does sound travel through a medium?",
    "What is ultrasound used for?",
    "What is the difference between loudness and pitch?",
    "What is SONAR?",
    "How does a tuning fork produce sound?"
]

retrieval_log = []

for query in eval_queries:
    results = retrieve(query, k=3)
    top1 = results[0]
    
    print(f"\nQ: {query}")
    print(f"Top-1: {top1['chunk_id']} | Score: {top1['score']} | Page: {top1['page']}")
    print(f"Preview: {top1['text'][:150]}")
    
    retrieval_log.append({
        "query": query,
        "top1_chunk_id": top1['chunk_id'],
        "top1_score": top1['score'],
        "top1_page": top1['page'],
        "top1_content_type": top1['content_type'],
        "top1_text_preview": top1['text'][:200],
        "top1_relevant": None  
    })

# Save to file
with open("retrieval_log.json", "w", encoding="utf-8") as f:
    json.dump(retrieval_log, f, indent=2, ensure_ascii=False)

print(f"\n Retrieval log saved with {len(retrieval_log)} entries")
print("Now manually review each result and fill in top1_relevant: YES or NO")


Q: What is the speed of sound in air?
Top-1: chunk_0064 | Score: 0.9002 | Page: 23
Preview: 206
Exploration|Grade 9
12. The speed of sound in air is about 331 m s–1 at 0 ºC and nearly 
344 m s–1 at 22 ºC. Roughly how much extra time will the 

Q: What is an echo?
Top-1: chunk_0050 | Score: 0.8501 | Page: 18
Preview: Sound Waves: Characteristics and Applications
201
10.7.1 Echo
If we shout near a mountain, a cliff, or in a long corridor we may hear our 
voice again

Q: How do humans hear sound?
Top-1: chunk_0000 | Score: 0.8725 | Page: 1
Preview: Sound Waves: 
Characteristics and 
Applications
Chapter 
10
Sound is an everyday sensory experience that helps us become 
aware of our surroundings. E

Q: What is the frequency of sound?
Top-1: chunk_0031 | Score: 0.9057 | Page: 11
Preview: 10.18 
indicating that it 
is almost a single 
frequency sound. λ
λ
λ
λ

Q: What is wavelength of a sound wave?
Top-1: chunk_0038 | Score: 0.9023 | Page: 14
Preview: Sound Waves: Characteristics and Applica

In [21]:
misses_content = """# Retrieval Misses Analysis

## Query 3: "How do humans hear sound?"
- **Top-1 chunk_id**: chunk_0000
- **Score**: 0.8725
- **Wrong chunk preview**: Chapter intro page — general overview of sound
- **Diagnosis**: Embedding miss — the query contains "humans hear" which 
  matched the intro page's general mention of everyday sounds. 
  The actual answer about auditory range is in chunk_0044 (rank 3).
  This is a **retrieval ranking failure** — right chunk exists but ranked 3rd.

## Query 9: "What is SONAR?"
- **Top-1 chunk_id**: chunk_0058  
- **Score**: 0.8744
- **Wrong chunk preview**: General sonar technology paragraph, no definition
- **Diagnosis**: Chunking miss — the SONAR definition and full explanation 
  got split across chunks. The top-1 has context around SONAR but not 
  the direct definition. This is a **chunk boundary failure**.

## Query 4: "What is the frequency of sound?" (partial)
- **Top-1 chunk_id**: chunk_0031
- **Score**: 0.9057
- **Issue**: Chunk contains garbled text with symbols (λλλλ) — 
  PDF extraction artifact from a figure page.
- **Diagnosis**: This is a **mixed structure failure** — figure labels 
  and symbols got mixed into the text chunk during extraction.
"""

with open("retrieval_misses.md", "w", encoding="utf-8") as f:
    f.write(misses_content)

print(" retrieval_misses.md saved")
print("\nMiss summary:")
print("  - Query 3: Ranking failure (right chunk at rank 3, not rank 1)")
print("  - Query 9: Chunk boundary failure (definition split across chunks)")
print("  - Query 4: Mixed structure failure (figure symbols in text)")

 retrieval_misses.md saved

Miss summary:
  - Query 3: Ranking failure (right chunk at rank 3, not rank 1)
  - Query 9: Chunk boundary failure (definition split across chunks)
  - Query 4: Mixed structure failure (figure symbols in text)


# Grounded Generation with Ollama
# Goal: Wire retriever to Ollama Gemma, add strict grounding

In [23]:
import requests

def call_ollama(prompt, model="gemma:7b"):
    response = requests.post(
        "http://localhost:11434/api/generate",
        json={
            "model": model,
            "prompt": prompt,
            "stream": False,
            "options": {"temperature": 0}
        }
    )
    return response.json()["response"]

# Quick test
test_response = call_ollama("Say exactly: Ollama is working.")
print(f"Ollama response: {test_response}")
print("Ollama connection working!")

Ollama response: Ollama is working.
Ollama connection working!


In [24]:
def ask_permissive(question):
    
    retrieved = retrieve(question, k=3)
    
    
    context = "\n\n".join([f"[{r['chunk_id']}]: {r['text']}" for r in retrieved])
    
  
    prompt = f"""Answer the question using the context below.

Context:
{context}

Question: {question}

Answer:"""
    
    answer = call_ollama(prompt)
    
    return {
        "question": question,
        "answer": answer,
        "chunk_ids": [r['chunk_id'] for r in retrieved]
    }

# Testing on 3 queries 
test_questions = [
    "What is the speed of sound in air?",
    "What is an echo?",
    "What is the speed of light?"   
]

print(" PERMISSIVE PROMPT RESULTS \n")
for q in test_questions:
    result = ask_permissive(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer'][:300]}")
    print(f"Chunks used: {result['chunk_ids']}")
    print()

 PERMISSIVE PROMPT RESULTS 

Q: What is the speed of sound in air?
A: The speed of sound in air is approximately **331 m s-1 at 0ºC** and **344 m s-1 at 22ºC**.
Chunks used: ['chunk_0064', 'chunk_0038', 'chunk_0036']

Q: What is an echo?
A: An echo is a sound that reflects off a distant object and comes back to us.
Chunks used: ['chunk_0050', 'chunk_0031', 'chunk_0058']

Q: What is the speed of light?
A: The provided text does not contain any information regarding the speed of light, so I am unable to answer this question from the given context.
Chunks used: ['chunk_0038', 'chunk_0067', 'chunk_0064']



In [25]:
def ask_strict(question):
    
    retrieved = retrieve(question, k=3)  
     
    print(f"--- Retrieved chunks for: '{question}' ---")
    for r in retrieved:
        print(f"  {r['chunk_id']} | score: {r['score']} | {r['text'][:80]}")
    print()    
    
    context = "\n\n".join([f"[{r['chunk_id']}]: {r['text']}" for r in retrieved])    
    
    prompt = f"""You are a study assistant for NCERT Science students.
Answer ONLY using the context below.
After every factual claim, cite the chunk it came from in square brackets, e.g. [chunk_0001].
If the answer is not present in the context, reply exactly: 'I don't have that in my study materials.'
Do not make up any information.

Context:
{context}

Question: {question}

Answer:"""
    
    answer = call_ollama(prompt)
    
    return {
        "question": question,
        "answer": answer,
        "chunk_ids": [r['chunk_id'] for r in retrieved]
    }

print("Strict ask function defined")

Strict ask function defined


In [26]:
test_questions = [
    "What is the speed of sound in air?",
    "What is an echo?",
    "What is the speed of light?"   # out of scope
]

print(" STRICT PROMPT RESULTS \n")
strict_results = []

for q in test_questions:
    result = ask_strict(q)
    print(f"Q: {result['question']}")
    print(f"A: {result['answer'][:300]}")
    print(f"Chunks used: {result['chunk_ids']}")
    print()
    strict_results.append(result)

 STRICT PROMPT RESULTS 

--- Retrieved chunks for: 'What is the speed of sound in air?' ---
  chunk_0064 | score: 0.9002 | 206
Exploration|Grade 9
12. The speed of sound in air is about 331 m s–1 at 0 ºC
  chunk_0038 | score: 0.8902 | Sound Waves: Characteristics and Applications
197
λ
speed of the wave
Therefore,
  chunk_0036 | score: 0.8864 | Therefore, 
the speed of sound (v ) is
distance
speed =  
time
                 

Q: What is the speed of sound in air?
A: The speed of sound in air is approximately 331 m s–1 at 0 ºC and 344 m s–1 at 22 ºC. [chunk_0036]
Chunks used: ['chunk_0064', 'chunk_0038', 'chunk_0036']

--- Retrieved chunks for: 'What is an echo?' ---
  chunk_0050 | score: 0.8501 | Sound Waves: Characteristics and Applications
201
10.7.1 Echo
If we shout near a
  chunk_0031 | score: 0.8342 | 10.18 
indicating that it 
is almost a single 
frequency sound. λ
λ
λ
λ
  chunk_0058 | score: 0.8276 | As technology 
improves, sound is becoming an even more powerful tool to explore

In [27]:
prompt_diff = """# Prompt Comparison: Permissive vs Strict

## Permissive Prompt
"Answer the question using the context below."

### Results:
- Speed of sound: Answered correctly (no citation)
- Echo: Answered correctly (no citation)  
- Speed of light (OOS): Refused — but by Gemma's own judgment, not our instruction.
  This is LUCKY behavior, not guaranteed. A different model or query might hallucinate.

---

## Strict Prompt
"Answer ONLY using the context below.
After every factual claim, cite the chunk in square brackets.
If the answer is not present, reply exactly: 
'I don't have that in my study materials.'"

### Results:
- Speed of sound: Answered correctly WITH citation [chunk_0036] 
- Echo: Answered correctly WITH citation [chunk_0050] 
- Speed of light (OOS): Refused with explanation 

---

## Key Insight
The permissive prompt relies on the model's judgment for refusal.
The strict prompt makes refusal a HARD CONSTRAINT — not optional behavior.
Result: citations now present on every factual claim, 
and out-of-scope refusal is enforced by instruction, not luck.
"""

with open("prompt_diff.md", "w", encoding="utf-8") as f:
    f.write(prompt_diff)

print("prompt_diff.md saved")

prompt_diff.md saved


In [28]:
def ask(question, k=3):
    
    retrieved = retrieve(question, k=k)
    
    
    context = "\n\n".join([f"[{r['chunk_id']}]: {r['text']}" for r in retrieved])
    
    # Strict prompt
    prompt = f"""You are a study assistant for NCERT Science students.
Answer ONLY using the context below.
After every factual claim, cite the chunk it came from in square brackets, e.g. [chunk_0001].
If the answer is not present in the context, reply exactly: 'I don't have that in my study materials.'
Do not make up any information.

Context:
{context}

Question: {question}

Answer:"""
    
    answer = call_ollama(prompt)
    
    return {
        "answer": answer,
        "sources": [f"Page {r['page']}" for r in retrieved],
        "chunk_ids": [r['chunk_id'] for r in retrieved]
    }

# Quick test
print("Testing ask() function...")
result = ask("What is an echo?")
print(f"\nAnswer: {result['answer']}")
print(f"Sources: {result['sources']}")
print(f"Chunk IDs: {result['chunk_ids']}")
print("\n ask() function ready!")

Testing ask() function...

Answer: An echo is a sound that reflects off a distant object and comes back to us. [chunk_0050]
Sources: ['Page 18', 'Page 11', 'Page 20']
Chunk IDs: ['chunk_0050', 'chunk_0031', 'chunk_0058']

 ask() function ready!
